# Capacity recommendation

Policy: **provisioned capacity = forecast + margin**, where the margin is a quantile of the decay model's rolling-origin residuals, per horizon band (steps 1–6, 7–12, 13–24). A larger margin lowers the risk of under-provisioning (demand above capacity) but leaves more capacity idle (cost).

In [1]:
import logging

import pandas as pd

from cwfc.baselines import decay_forecast
from cwfc.evaluation import backtest
from cwfc.interval import STEP_BANDS
from cwfc.split import split_holdout
from cwfc.targets import build_targets

logging.basicConfig(level=logging.INFO)
readings = pd.read_csv("../challenge_data/vm_cpu_readings.csv.gz")
targets = build_targets(readings)
train, holdout_timestamps = split_holdout(targets)
TARGET = "aggregate_cpu_utilization"

edges = [0] + [end for _, end in STEP_BANDS]
labels = [f"{start}-{end}" for start, end in STEP_BANDS]

folds = backtest(train, decay_forecast, TARGET)
folds["resid"] = folds["actual"] - folds["forecast"]
folds["band"] = pd.cut(folds["step"], edges, labels=labels)

template = pd.DataFrame({"timestamp": holdout_timestamps})
hold = pd.DataFrame({
    "step": range(1, 25),
    "forecast": decay_forecast(train, template, TARGET).to_numpy(),
    "actual": targets.set_index("timestamp").loc[holdout_timestamps.to_numpy(), TARGET].to_numpy(),
})
hold["band"] = pd.cut(hold["step"], edges, labels=labels)

INFO:cwfc.targets:Dropped 0 of 1340000 rows


## Trade-off between under- and over-provisioning

`under %` = share of 5-minute steps where actual demand exceeded provisioned capacity. `headroom` = average idle capacity (CPU points). Fold numbers are in-sample for the margin (the quantiles are estimated on the same residuals), so the holdout columns are the independent check.

In [2]:
def evaluate(frame, offsets):
    capacity = frame["forecast"] + frame["band"].map(offsets).astype(float)
    return (frame["actual"] > capacity).mean() * 100, (capacity - frame["actual"]).mean()

rows = []
for name, q in [("no margin", None), ("q80", 0.80), ("q90", 0.90), ("q95", 0.95), ("max residual", 1.0)]:
    if q is None:
        offsets = pd.Series(0.0, index=labels)
    else:
        offsets = folds.groupby("band", observed=True)["resid"].quantile(q)
    fold_under, fold_head = evaluate(folds, offsets)
    hold_under, hold_head = evaluate(hold, offsets)
    rows.append({"policy": name, "fold under %": fold_under, "fold headroom": fold_head,
                 "holdout under %": hold_under, "holdout headroom": hold_head})
pd.DataFrame(rows).set_index("policy").round(3)

,fold under %,fold headroom,holdout under %,holdout headroom
policy,,,,
no margin,36.905,0.010,25.000,0.124
q80,20.833,0.233,4.167,0.346
q90,11.310,0.334,0.000,0.448
q95,6.548,0.423,0.000,0.536
max residual,0.000,0.526,0.000,0.640


## Chosen margin (q90) per band

In [3]:
q90 = folds.groupby("band", observed=True)["resid"].quantile(0.90)
print("q90 margin per band (CPU points):", q90.round(3).to_dict())
print("as % of the training level:", (q90 / train[TARGET].mean() * 100).round(1).to_dict())

q90 margin per band (CPU points): {'1-6': 0.389, '7-12': 0.417, '13-24': 0.244}
as % of the training level: {'1-6': 4.1, '7-12': 4.4, '13-24': 2.6}
